### TEL328 - Procesamiento Digital de Imágenes
# Taller 1

Nombre:

## Reglas
1. **Fecha de entrega**:
    - El plazo límite de entrega vence el dia viernes 29/08/2025 a las 23:59.
    - Descuento por atraso: 10 puntos por cada hora.
2. Entregable:
    - La tarea debe ser entregada en un archivo .zip con el formato ```Taller1_Nombre_Apellido.zip```
    - Se debe incluir: 
        - Notebook en formato ```Taller1_Nombre_Apellido.ipynb``` con todo el desarrollo.
        - Imágenes usadas y generadas
        
>**En caso de no cumplir con el formato, se descontará 10 puntos.**

## Introducción

En este taller trabajaremos con un tipo de ruido **no convencional** que degrada severamente la imagen: parches negros colocados en posiciones aleatorias.  

Este tipo de daño **no** se puede resolver eficazmente con filtros lineales o no lineales básicos, ya que no se trata de un ruido aditivo leve, sino de **pérdida total de información** en regiones completas.

El objetivo es diseñar e implementar un **pipeline de restauración (*inpainting*)** usando clustering con **K-Means** para reconstruir las zonas dañadas de manera coherente con su contexto local.

Se espera que:  
- Sepa utilizar librerías estándar para filtrado de imágenes.  
- Diseñe una estrategia de reconstrucción usando K-Means combinando **color** y **posición** como características.  
- Evalúe cuantitativamente y cualitativamente los resultados.  
- Reflexione sobre ventajas, limitaciones y mejoras posibles.

Importaciones

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

### 1) Generar el ruido de cuadrados negros

Se usará el siguiente generador de ruido:

In [ ]:
def add_random_squares(image, num_squares, min_size, max_size):
    noisy_image = np.copy(image)
    for _ in range(num_squares):
        size = np.random.randint(min_size, max_size + 1)
        x = np.random.randint(0, image.shape[1] - size)
        y = np.random.randint(0, image.shape[0] - size)
        noisy_image[y:y+size, x:x+size] = 0
    return noisy_image

Tareas:

- Elige una imagen a color (JPG o PNG).

- Genera tres versiones ruidosas con diferentes parámetros.

- Comenta brevemente las diferencias visuales entre cada configuración.

In [ ]:
# Completar con su código. Puede agregar todas las celdas que desee.

*Para el análisis, utilice celdas Markdown.*

### 2) Intento de limpieza con filtros de librería

En esta sección probarás filtros **tradicionales de suavizado** disponibles en **OpenCV**, con el objetivo de observar cómo responden frente al ruido disruptivo introducido en la imagen.

---

#### 2.1. Filtros a implementar
Debes aplicar **al menos 3 filtros distintos**, probando **dos tamaños de ventana** (pequeño y grande) para cada uno. Los filtros sugeridos son:

- **Filtro de media**: `cv2.blur`  
- **Filtro gaussiano**: `cv2.GaussianBlur`  
- **Filtro de mediana**: `cv2.medianBlur`  
- **Filtro bilateral**: `cv2.bilateralFilter`  

> Puedes elegir cuáles aplicar, pero deben ser al menos 3 de la lista.

---

#### 2.2. Comparativa visual
Para cada imagen dañada, genera una figura con **subplots** mostrando:

- Imagen original.  
- Imagen dañada.  
- Resultados con cada filtro (dos versiones: ventana pequeña y ventana grande).  

Ejemplo de layout esperado:

| Original | Dañada | Media (5×5) | Media (15×15) |
|----------|--------|-------------|---------------|
| Gauss (5×5) | Gauss (15×15) | Mediana (5) | Mediana (15) |

---

#### 2.3. Evaluación con MAE
- Calcula el **MAE (Mean Absolute Error)** entre la imagen original y cada resultado filtrado, **solo en la región dañada** definida por la máscara.  
- Registra los valores de MAE en una tabla o imprime un resumen en consola.  
- Identifica cuál fue el **mejor filtro** (menor MAE).  

---

#### 2.4. Discusión crítica
- Explica **por qué los filtros tradicionales no funcionan bien** en este escenario.

- Responde las siguientes preguntas:
  - ¿Qué filtro fue más robusto frente a los cuadrados negros?  
  - ¿Qué efecto tuvo aumentar el tamaño de la ventana?  
  - ¿Se observó algún trade-off entre calidad visual y error numérico (MAE)?  

In [ ]:
# Completar con su código. Puede agregar todas las celdas que desee.

*Para el análisis, utilice celdas Markdown.*

### 3) Restauración por inpainting con K-Means

En esta sección deberás implementar un algoritmo de **inpainting** basado en **clustering K-Means**, con el objetivo de reconstruir las zonas dañadas por cuadrados negros. 

El proceso se divide en varias etapas:

---

#### 3.1. Crear máscara de daño
- Detecta los píxeles negros `(0,0,0)` en la imagen dañada.  
- Construye una **máscara binaria** donde `1` indica píxeles dañados y `0` píxeles intactos.  
- Esta máscara será usada para distinguir qué píxeles entran al entrenamiento y cuáles deben reconstruirse.

---

#### 3.2. Espacio de características
- Convierte la imagen al espacio de color **Lab** (mejor para representar similitud perceptual).  
- Para cada píxel, crea un vector de características de la forma:  

  $$
  f = [L, a, b, \lambda x, \lambda y]
  $$

  donde:
  - `(L,a,b)` corresponden a los valores Lab.  
  - `(x,y)` son las coordenadas normalizadas entre 0 y 1.  
  - `λ` es un factor de ponderación entre `0.1` y `0.5` que controla cuánto influye la posición espacial.  

- Ajusta `λ` y observa cómo cambia la reconstrucción.

---

#### 3.3. Entrenar K-Means solo con píxeles intactos
- Extrae únicamente los **píxeles intactos** (los que no son negros).  
- Aplica **K-Means** para agrupar estos píxeles en `k` clusters.  
- Prueba al menos dos valores de `k` (ejemplo: `k=4` y `k=8`).  
- Cada cluster tendrá un **centroide Lab**, que representa un color promedio de una región de la imagen.

---

#### 3.4. Rellenar zonas dañadas
- Para cada píxel dañado:  
  - Define una **ventana local** (ej. 7×7, 11×11, etc.).  
  - Determina el **cluster dominante** en la vecindad (ignora los píxeles dañados).  
  - Asigna al píxel el **color del centroide** de ese cluster.  

- Si no hay suficientes vecinos válidos, asigna el cluster más frecuente de toda la imagen.

---

#### 3.5. Suavizado de bordes
- Una vez reconstruida la imagen, aplica un **suavizado en la zona de transición** (por ejemplo, con un **filtro bilateral**) para reducir artefactos.  
- El suavizado debe aplicarse principalmente en los bordes entre zonas intactas y reconstruidas.

---

#### 3.6. Evaluar resultados
- Usa la **métrica MAE (Mean Absolute Error)**, restringida a la región dañada definida por la máscara.  

- Compara:
  - El **mejor filtro** de la sección 2 (el que haya dado menor MAE).  
  - El **inpainting con K-Means**.  

- Presenta los resultados de dos formas:
  1. **Tabla en consola** con los valores de MAE para cada caso.  
  2. **Subplots comparativos** mostrando:  
     - Imagen original.  
     - Imagen dañada.  
     - Resultado del mejor filtro.  
     - Resultado del inpainting con K-Means.  

  Ejemplo de mosaico esperado:  

  | Original | Dañada | Mejor filtro | K-Means |
  |----------|--------|--------------|---------|

- Discute cuál método logra menor error y si visualmente corresponde con la calidad percibida.

---

#### 3.7. Discusión crítica
- Reflexiona sobre:
  - ¿Qué valor de `k` funcionó mejor y por qué?  
  - ¿Cómo influyó el parámetro `λ` en la reconstrucción?  
  - ¿Qué limitaciones tiene este método frente a técnicas de inpainting avanzadas?  
  - ¿Qué mejoras podrías proponer? (ejemplo: usar información de texturas, interpolación espacial más sofisticada, redes neuronales, etc.).  


In [ ]:
# Completar con su código. Puede agregar todas las celdas que desee.

*Para el análisis, utilice celdas Markdown.*

In [ ]:
# Completar con su código. Puede agregar todas las celdas que desee.

*Para el análisis, utilice celdas Markdown.*

### Ejemplo visual del problema a resolver

La siguiente figura muestra un ejemplo de la situación a abordar:

| Imagen original | Imagen dañada (ruido de cuadrados negros) | Ejemplo de restauración |
|-----------------|--------------------------------------------|--------------------------|
| ![Original](original.jpg) | ![Dañada](danada.jpg) | ![Restaurada](restaurada.jpg) |

- **Original:** Imagen sin alteraciones.
- **Dañada:** Ruido disruptivo por cuadrados negros aleatorios.
- **Restaurada:** Ejemplo ilustrativo de reconstrucción (*inpainting*) basada en información de la propia imagen.

> Nota: El resultado mostrado es solo una referencia conceptual. Tu implementación deberá producir su propia versión restaurada, que puede diferir en detalles y calidad.


### Entregables

- Imágenes original y ruidosas.

- Resultados de filtros de librería con análisis.

- Implementación y resultados del pipeline K-Means con visualizaciones intermedias.

- Métricas cuantitativas.

- Conclusiones.